# Chapitre 4 — Raisonner avec des probabilités · Corrigé commenté

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch04_corrige.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Chaque réponse commence par le raisonnement sur papier, puis montre son équivalent en Python. Les commentaires signalent, à chaque calcul, le groupe qui sert de dénominateur.

**Sommaire.** Données · Les calculs des leçons 4.1 à 4.5 · Exercices 4.1 à 4.6

**Avant chaque calcul**, demandez-vous sur quel ensemble vous comptez : des spams, des messages signalés ou des messages reçus.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur = un rôle
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLEU, ORANGE = '#2a78d6', '#eb6834'   # « aucun spam manqué », courbe de la loi normale
from math import comb

## Données

Les identifiants D4.1 et D4.2 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 4, recopiées des fichiers CSV du livre (ne pas modifier)
# Lignes : spam, légitime. Colonnes : signalé, non signalé.
LIGNES, COLONNES = ['spam', 'légitime'], ['signalé', 'non signalé']

# D4.1 — boîte où 20 % des messages sont des spams (1 000 messages)
boite_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=LIGNES, columns=COLONNES)

# D4.2 — même filtre, boîte où 2 % des messages sont des spams (1 000 messages)
boite_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=LIGNES, columns=COLONNES)

# D4.3 — 1 000 durées de traitement (ms), régénérées à l'identique du fichier CSV
durees = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 4.1 à 4.5.

In [ ]:
# Leçon 4.1 — compter, puis diviser par le bon groupe (équation 4.1)
t = boite_20
n = t.values.sum()
print('total :', n, '| spams :', t.loc['spam'].sum(), '| signalés :', t['signalé'].sum())
print('P(spam) =', t.loc['spam'].sum() / n, '| P(signalé) =', t['signalé'].sum() / n,
      '| P(spam et signalé) =', t.loc['spam', 'signalé'] / n)
# Parmi les signalés : le dénominateur est la colonne « signalé »
print(f"spams parmi les signalés : {t.loc['spam', 'signalé']} / {t['signalé'].sum()} "
      f"≈ {t.loc['spam', 'signalé'] / t['signalé'].sum():.2f}")

In [ ]:
# Leçon 4.2 — le même filtre dans une boîte où les spams sont rares (équation 4.2)
d, f = 0.90, 0.05                   # parts signalées : des spams, des messages légitimes
for p, t in [(0.20, boite_20), (0.02, boite_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    reconstruit = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(reconstruit) == t.values).all()   # le tableau se déduit des taux
    print(f'taux de base {p:.0%} : {t.values.tolist()}  →  part de spams parmi les signalés '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Leçon 4.3 — deux probabilités conditionnelles, deux dénominateurs (équation 4.3)
def conditionnelle(t, case, groupe):
    """Part de la case (ligne, colonne) dans le groupe : une ligne ('spam') ou une colonne ('signalé')."""
    denominateur = t.loc[groupe].sum() if groupe in t.index else t[groupe].sum()
    return t.loc[case] / denominateur, denominateur

for nom, t in [('20 %', boite_20), ('2 %', boite_2)]:
    rappel, den_r = conditionnelle(t, ('spam', 'signalé'), 'spam')
    precision, den_p = conditionnelle(t, ('spam', 'signalé'), 'signalé')
    print(f'boîte à {nom} : P(signalé | spam) = {rappel:.2f} (dénominateur {den_r}) ; '
          f'P(spam | signalé) = {precision:.2f} (dénominateur {den_p})')

# Les mêmes nombres avec scikit-learn, à partir d'étiquettes message par message (s'il est installé)
try:
    from sklearn.metrics import precision_score, recall_score
    vrai = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = spam
    predit = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = signalé
    print(f'scikit-learn : rappel = {recall_score(vrai, predit):.2f}, précision = {precision_score(vrai, predit):.2f}')
except ImportError:
    print('scikit-learn absent : ce calcul est facultatif.')

In [ ]:
# Leçon 4.4 — le théorème de Bayes (équations 4.4 et 4.5) et la courbe du livre
def bayes(p, d=0.90, f=0.05):
    p_signale = d * p + f * (1 - p)                  # équation 4.5
    return d * p / p_signale                        # équation 4.4

for p in (0.20, 0.02):
    print(f'P(spam) = {p} : P(signalé) = {0.9 * p + 0.05 * (1 - p):.3f}, P(spam | signalé) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=SIGNALE, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Taux de base : part de spams (%)'); plt.ylabel('P(spam | signalé)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Leçon 4.5 — dix spams : loi binomiale (équation 4.7) et événement complémentaire (équation 4.8)
n, p = 10, 0.1                        # on compte sur 10 SPAMS ; p = probabilité d'être manqué
loi = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0,9^10 =', round(0.9 ** 10, 4), '| au moins un manqué =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 à 4 :', [round(v, 3) for v in loi[:5]], '| somme =', round(sum(loi), 6))
couleurs = [BLEU] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), loi, color=couleurs, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Nombre de spams manqués parmi 10 spams'); plt.ylabel('Probabilité'); plt.show()

# Aperçu de la loi normale : durées de traitement (D4.3)
m, s = durees.mean(), durees.std()
print(f'moyenne = {m:.2f} ms, écart-type = {s:.2f} ms ; à ±1 écart-type : {np.mean(abs(durees - m) <= s):.1%}, '
      f'à ±2 écarts-types : {np.mean(abs(durees - m) <= 2 * s):.1%}')

## Exercice 4.1 — Lire un graphique : vingt spams

*Notions : loi binomiale, événement complémentaire (leçon 4.5).* Lisez d'abord la figure du livre.

### a) à d) Corrigé commenté

On compte ici sur **20 spams**, pas sur des messages reçus.

**a)** Aucun spam manqué : environ 0,36. Exactement un : environ 0,38.

**b)** La barre la plus haute est celle de 1 : un spam manqué est le cas le plus probable.

**c)** $1 - 0{,}36 \approx 0{,}64$. Avec l'équation 4.8 : $1 - 0{,}95^{20} \approx 0{,}6415$.

**d)** Chaque spam a deux fois moins de chances de passer (5 % au lieu de 10 %), mais il y a deux fois plus de spams (20 au lieu de 10). En moyenne, le filtre en manque toujours un : $20 \times 0{,}05 = 1$. La probabilité qu'au moins un passe reste donc proche : 64 % contre 65 %.

In [ ]:
def binomiale(n, p, k):
    return comb(n, k) * p ** k * (1 - p) ** (n - k)

In [ ]:
# Vérification sur les 20 spams de l'exercice
if binomiale(20, 0.05, 0) is None:
    print('Complétez la fonction.')
else:
    p0, p1 = binomiale(20, 0.05, 0), binomiale(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, au moins un = {1 - p0:.4f} →',
          'correct' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'à revoir')

## Exercice 4.2 — Un filtre plus prudent

*Notions : probabilité conditionnelle (leçon 4.3), théorème de Bayes (leçon 4.4), taux de base (leçon 4.2).* Le filtre prudent signale 75 % des spams et 1 % des messages légitimes. Boîte de 10 000 messages, dont 10 % de spams. Faites d'abord les questions a) à e) sur papier.

### a) à c) Corrigé commenté

**a)** Commencez par les lignes, c'est-à-dire par ce que sont les messages. 10 % de 10 000 donnent **1 000 spams**, et il reste **9 000 messages légitimes**. Le filtre signale 75 % des spams : 0,75 × 1 000 = **750**, donc 250 spams passent. Il signale 1 % des messages légitimes : 0,01 × 9 000 = **90** fausses alertes, et 8 910 messages légitimes passent.

| | Signalé | Non signalé | Total |
|:--|--:|--:|--:|
| Spam | 750 | 250 | 1 000 |
| Légitime | 90 | 8 910 | 9 000 |
| **Total** | **840** | **9 160** | **10 000** |

**b)** $P(\text{signalé} \mid \text{spam})$ : on compte **dans la ligne des spams**, qui sert de dénominateur. $750 / 1\,000 = 0{,}75$. On retrouve le taux annoncé, ce qui est rassurant. $P(\text{spam} \mid \text{signalé})$ : on compte **dans la colonne des signalés**. $750 / 840 \approx 0{,}893$. Même numérateur, 750 ; dénominateurs différents, 1 000 et 840.

**c)** Avec l'équation 4.5 : $P(\text{signalé}) = 0{,}75 \times 0{,}1 + 0{,}01 \times 0{,}9 = 0{,}075 + 0{,}009 = 0{,}084$, soit les 840 messages signalés sur 10 000. Avec l'équation 4.4 : $P(\text{spam} \mid \text{signalé}) = 0{,}075 / 0{,}084 \approx 0{,}893$. Bayes ne fait que réécrire le comptage du tableau avec des proportions.

### a) Le tableau en Python

In [ ]:
spam_signale = round(0.75 * 1000)          # 750 : 75 % des 1 000 spams
spam_non_signale = 1000 - spam_signale     # 250
legitime_signale = round(0.01 * 9000)      # 90 fausses alertes : 1 % des 9 000 légitimes
legitime_non_signale = 9000 - legitime_signale   # 8 910

In [ ]:
# Vérification
cases = [spam_signale, spam_non_signale, legitime_signale, legitime_non_signale]
if None in cases:
    print('Complétez la cellule précédente.')
else:
    prudent = pd.DataFrame([cases[:2], cases[2:]], index=LIGNES, columns=COLONNES)
    print(prudent)
    print('tableau :', 'correct' if cases == [750, 250, 90, 8910] else 'à revoir')

### b) Les deux probabilités conditionnelles

In [ ]:
p_signale_si_spam = 750 / (750 + 250)   # dénominateur : la ligne des spams
p_spam_si_signale = 750 / (750 + 90)    # dénominateur : la colonne des signalés

In [ ]:
# Vérification
for nom, la_votre, vrai in [('P(signalé | spam)', p_signale_si_spam, 0.75),
                            ('P(spam | signalé)', p_spam_si_signale, 750 / 840)]:
    print(f'{nom} :', 'pas encore complété' if la_votre is None else
          ('correct' if abs(la_votre - vrai) < 0.001 else 'à revoir'))

### c) à e) Le théorème de Bayes en fonction

In [ ]:
def proba_spam_si_signale(p, d, f):
    p_signale = d * p + f * (1 - p)     # équation 4.5 : les deux cases de la colonne « signalé »
    return d * p / p_signale            # équation 4.4 : la case rouge divisée par la colonne

In [ ]:
# Vérification de c), d) et e)
if proba_spam_si_signale(0.1, 0.75, 0.01) is None:
    print('Complétez la fonction.')
else:
    cas = [('c) prudent, 10 % de spams', (0.10, 0.75, 0.01), 0.8929),
           ('d) prudent, 1 % de spams', (0.01, 0.75, 0.01), 0.4310),
           ('e) prudent, 2 % de spams', (0.02, 0.75, 0.01), 0.6048),
           ('e) filtre des leçons, 2 % de spams', (0.02, 0.90, 0.05), 0.2687)]
    for nom, args, attendu in cas:
        v = proba_spam_si_signale(*args)
        print(f'{nom} : {v:.4f} →', 'correct' if abs(v - attendu) < 0.0001 else 'à revoir')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, proba_spam_si_signale(P, 0.90, 0.05), color=SIGNALE, linewidth=2,
             label='filtre des leçons (90 % ; 5 %)')
    plt.plot(100 * P, proba_spam_si_signale(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='filtre prudent (75 % ; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Taux de base : part de spams (%)'); plt.ylabel('P(spam | signalé)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) et e) Corrigé commenté

**d)** Prédiction attendue : la probabilité **diminue**, puisque les spams deviennent plus rares et que les fausses alertes pèsent davantage (leçon 4.2). Pour 10 000 messages : 100 spams, dont 75 signalés ; 9 900 messages légitimes, dont 99 signalés. Dans la colonne des signalés : $75 / (75 + 99) = 75 / 174 \approx 0{,}431$. Moins d'un signalement sur deux est un vrai spam, alors que le filtre n'a pas changé.

**e)** Pour le filtre prudent dans la boîte à 2 % : $\dfrac{0{,}75 \times 0{,}02}{0{,}75 \times 0{,}02 + 0{,}01 \times 0{,}98} = \dfrac{0{,}015}{0{,}0248} \approx 0{,}605$, contre 0,27 pour le filtre des leçons. Comparer ces deux nombres, c'est comparer la **précision** des deux filtres : on compte dans la colonne des messages signalés. Quand le filtre prudent signale un message, on peut davantage s'y fier : il fait cinq fois moins de fausses alertes. Mais pour choisir un filtre, la précision ne suffit pas. Comptez aussi dans la ligne des spams : sur 100 spams, le filtre prudent en manque 25, le filtre des leçons 10 (son **rappel** vaut 0,90 contre 0,75). Le bon choix dépend de ce qui coûte le plus : un message important perdu dans « Indésirables », ou un spam de plus dans la boîte de réception. Comme au chapitre 3, il faut décider de la mesure avant de comparer. La courbe montre que l'écart entre les deux filtres est le plus grand quand les spams sont rares.

## Exercice 4.3 — Trois spams, puis dix messages

*Notions : épreuve de Bernoulli et loi binomiale (leçon 4.5), probabilité conditionnelle (leçon 4.3).* Faites d'abord les questions sur papier.

### a) à e) Corrigé commenté

**a)** Chaque branche multiplie 0,9 (signalé) ou 0,1 (manqué). L'issue « trois signalés » vaut $0{,}9^3 = 0{,}729$ ; chaque issue avec un seul spam manqué vaut $0{,}1 \times 0{,}9^2 = 0{,}081$ ; avec deux manqués, $0{,}1^2 \times 0{,}9 = 0{,}009$ ; les trois manqués, $0{,}1^3 = 0{,}001$.

**b)** Trois issues ont un seul spam manqué : $P(X = 1) = 3 \times 0{,}081 = 0{,}243$. De même $P(X = 2) = 3 \times 0{,}009 = 0{,}027$, $P(X = 0) = 0{,}729$, $P(X = 3) = 0{,}001$. Somme : $0{,}729 + 0{,}243 + 0{,}027 + 0{,}001 = 1$. Le facteur 3 est le coefficient $\binom{3}{1} = \binom{3}{2} = 3$.

**c)** En additionnant : $0{,}243 + 0{,}027 + 0{,}001 = 0{,}271$. Par le complémentaire : $1 - 0{,}729 = 0{,}271$. Le second calcul est plus court.

**d)** **Vérifiez sur quel ensemble vous comptez.** On ne compte plus sur dix *spams*, mais sur dix *messages reçus*, dont certains sont légitimes. Pour un message reçu, « spam qui passe » veut dire « spam **et** non signalé » : $0{,}2 \times 0{,}1 = 0{,}02$ (leçon 4.3). Aucun des dix ne passe avec la probabilité $0{,}98^{10} \approx 0{,}817$, donc au moins un spam passe avec la probabilité $1 - 0{,}98^{10} \approx 0{,}183$.

**e)** Dans la leçon 4.5, les dix messages sont tous des spams : chacun a 10 % de chances de passer. Parmi dix messages reçus, il n'y a en moyenne que deux spams, et chaque message n'a que 2 % de chances d'être un spam qui passe. « Dix spams » et « dix messages reçus » sont deux questions différentes.

In [ ]:
loi_3 = [comb(3, k) * 0.1 ** k * 0.9 ** (3 - k) for k in range(4)]   # sur 3 SPAMS
p_d = 1 - (1 - 0.2 * 0.1) ** 10   # sur 10 MESSAGES REÇUS : spam ET non signalé = 0,02

In [ ]:
# Vérification
if loi_3 is None or p_d is None:
    print('Complétez la cellule précédente.')
else:
    print('b) :', [round(v, 3) for v in loi_3], '→', 'correct' if np.allclose(loi_3, [0.729, 0.243, 0.027, 0.001]) else 'à revoir')
    print(f'd) : {p_d:.4f} →', 'correct' if abs(p_d - 0.1829) < 0.0001 else 'à revoir')

## Exercice 4.4 — Simuler dix spams

*Notions : loi binomiale (leçon 4.5), fréquence et probabilité.* On compte sur **dix spams**.

In [ ]:
def simuler_serie(rng):
    """Renvoie le nombre de spams manqués parmi 10. rng.random() tire un nombre entre 0 et 1."""
    return int(np.sum(rng.random(10) < 0.1))   # un tirage par spam : manqué si < 0,1

In [ ]:
# b) et c) 10 000 séries (graine fixée pour retrouver les mêmes nombres)
rng = np.random.default_rng(4)
if simuler_serie(rng) is None:
    print('Complétez la fonction.')
else:
    rng = np.random.default_rng(4)
    X = np.array([simuler_serie(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'fréquence observée : {freq:.4f} ; probabilité calculée : {1 - 0.9 ** 10:.4f} →',
          'correct' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'à revoir')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=SIGNALE, label='fréquence observée')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='probabilité (équation 4.7)')
    plt.xlabel('Spams manqués parmi 10'); plt.legend(); plt.show()
    # d) Plus de séries : la fréquence se rapproche de la probabilité
    for N in (100, 100, 100, 100000):
        Y = np.array([simuler_serie(rng) for _ in range(N)])
        print(f'{N:>6} séries : fréquence = {np.mean(Y >= 1):.4f}')

### Corrigé commenté

Avec la graine du notebook, 10 000 séries donnent une fréquence de 0,6549, pour une probabilité de 0,6513. Les deux sont proches, mais pas égales : une fréquence observée varie d'une simulation à l'autre. Avec 100 séries, elle peut s'écarter de plusieurs points de pourcentage (relancez la cellule sans graine pour le voir). Avec 100 000 séries, elle s'en approche à quelques millièmes près. La probabilité est la valeur vers laquelle tend la fréquence quand le nombre de séries augmente. Vérifiez aussi l'ensemble : chaque série compte des **spams**, jamais des messages reçus.

## Exercice 4.5 — Prédire, puis vérifier : une campagne de spams

*Notions : indépendance, événement complémentaire (leçon 4.5).*

### a) à c) et e) Corrigé commenté

**b)** Le filtre manque les dix spams d'un coup avec la probabilité 0,1, ou n'en manque aucun. Le nombre de spams manqués vaut donc 0 ou 10. Au moins un passe avec la probabilité **0,1**, bien moins que 65 %.

**c)** En moyenne : $10 \times 0{,}1 + 0 \times 0{,}9 = 1$ spam manqué. C'est la même moyenne que dans la leçon 4.5 ($10 \times 0{,}1 = 1$). La moyenne est identique, mais la répartition est très différente.

**e)** L'indépendance n'est plus vérifiée : savoir que le premier spam est manqué rend les autres presque sûrement manqués. La probabilité d'être manqué, 0,1 pour chacun, n'a pas changé. Pour l'utilisateur, dix spams d'un coup sont plus gênants, et plus visibles. La loi binomiale ne s'applique qu'avec ses deux hypothèses.

In [ ]:
def simuler_campagne(rng):
    return 10 if rng.random() < 0.1 else 0   # un seul tirage pour toute la campagne

In [ ]:
# d) Vérification par simulation
rng = np.random.default_rng(5)
if simuler_campagne(rng) is None:
    print('Complétez la fonction.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([simuler_campagne(rng) for _ in range(10000)])
    print(f'au moins un manqué : {np.mean(Z >= 1):.4f} ; moyenne : {Z.mean():.3f} →',
          'correct' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'à revoir')

## Exercice 4.6 — Défi : combien de spams pour dépasser 99 % de risque d'en manquer un ?

*Notions : lois de Bernoulli et binomiale, loi normale (leçon 4.5), écart-type (leçon 1.1).*

### a) à d) Corrigé commenté

**a)** Notons $X_i = 1$ si le spam $i$ est manqué, 0 sinon. Alors $X = X_1 + \dots + X_n$ : additionner les 1 compte les spams manqués. Chaque $X_i$ vaut en moyenne $1 \times p + 0 \times (1 - p) = p$, et la moyenne d'une somme est la somme des moyennes : $n\,p$. Pour 10 spams, 1.

**b)** Il faut $1 - 0{,}9^n > 0{,}99$, soit $0{,}9^n < 0{,}01$. On trouve $0{,}9^{43} \approx 0{,}0108$ et $0{,}9^{44} \approx 0{,}0097$ : il faut **44 spams**.

**c)** Moyenne : $100 \times 0{,}1 = 10$. Écart-type : $\sqrt{100 \times 0{,}1 \times 0{,}9} = \sqrt{9} = 3$.

**d)** Les barres de la loi binomiale épousent presque la courbe en cloche de moyenne 10 et d'écart-type 3. Quand $n$ est grand, la loi binomiale ressemble à une loi normale : c'est pourquoi la courbe de la leçon 4.5 reviendra souvent.

In [ ]:
n_min = next(n for n in range(1, 200) if 0.9 ** n < 0.01)   # on essaie n = 1, 2, 3…
moyenne_100 = 100 * 0.1
ecart_type_100 = np.sqrt(100 * 0.1 * 0.9)

In [ ]:
# Vérification, puis d) binomiale contre loi normale
if None in (n_min, moyenne_100, ecart_type_100):
    print('Complétez la cellule précédente.')
else:
    print(f'n_min = {n_min} →', 'correct' if n_min == 44 else 'à revoir')
    print(f'moyenne = {moyenne_100:.1f}, écart-type = {ecart_type_100:.1f} →',
          'correct' if np.isclose(moyenne_100, 10) and np.isclose(ecart_type_100, 3) else 'à revoir')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='loi binomiale (n = 100, p = 0,1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='loi normale (moyenne 10, écart-type 3)')
    plt.xlabel('Spams manqués parmi 100'); plt.legend(); plt.show()